In [1]:
from src.classes.crossword_puzzle import CrosswordPuzzle
from src.classes.guesses import Guess
from src.constants import CLUE_ID
from src.prompts.get_clue_difficulty_with_llm import get_clue_difficulty_with_llm
from src.prompts.get_guesses_for_clue_using_llm import get_guesses_for_clue_using_llm


In [2]:

file = "southern crosswords mini 195.puz"
crossword_puzzle = CrosswordPuzzle(file)
guesses: dict[CLUE_ID, list[Guess]] = {}
completed_clues: list[CLUE_ID] = []

In [3]:
clue_difficulties = get_clue_difficulty_with_llm(crossword_puzzle.get_clues(), debug=True)

=== REORDER CLUES with llama3.1:latest (Attempts: 1) ===

You are a crossword puzzle solver. You are given a list of crossword clues with the goal of assigning a vagueness score and a complexity score to each.
Assign each clue a vagueness score from 0 to 100, with 0 being the least vague and 100 being the most vague.
Assign each clue a complexity score from 0 to 100, with 0 being the least complex and 100 being the most complex.
Provide a brief explanation of why each clue is considered to have the given vagueness and complexity scores,including any wordplay, obscurity, or other factors that contribute to its difficulty.
At the end of the explanation, provide some potential answers that could fit the clue, which can help illustrate the vagueness and complexity of the clue.
Clues are less vague if they have only one obvious answer, while more vague clues have multiple plausible answers or interpretations, making them harder to solve.
Clues are more complex when they require multiple ste

In [ ]:
while not crossword_puzzle.is_solved:
    crossword_puzzle.print_grid()

    number_of_known_letters = crossword_puzzle.get_number_of_known_letters_for_all_clues()

    print("\n=== Solver iteration ===")
    print("Known letters per clue:")
    for clue_id, known_count in sorted(number_of_known_letters.items()):
        difficulty = clue_difficulties.get(clue_id, "?")
        print(f"  {clue_id}: known={known_count}, difficulty={difficulty}")

    ranked_clues = sorted(
        crossword_puzzle.incomplete_clues,
        key=lambda c: (-number_of_known_letters[c.id], clue_difficulties[c.id]),
    )

    print("\nRanked incomplete clues (best first):")
    for i, c in enumerate(ranked_clues, start=1):
        print(
            f"  {i}. {c.number} {c.direction:<6} "
            f"(known={number_of_known_letters[c.id]}, difficulty={clue_difficulties[c.id]}) - {c.text}"
        )

    clue = ranked_clues[0]
    print(
        f"\nSelected clue -> {clue.number} {clue.direction} "
        f"(known={number_of_known_letters[clue.id]}, difficulty={clue_difficulties[clue.id]})"
    )

    print(f"Attempting to solve clue {clue.number} {clue.direction} - {clue.text}")

    print("guesses dict:")
    print(guesses)

    if clue.id not in guesses:
        pattern = crossword_puzzle.get_pattern(clue)
        print(f"Generating guesses for clue: {clue.number} {clue.direction} - {clue.text} with pattern '{pattern}'")
        clue_guesses = get_guesses_for_clue_using_llm(clue ,pattern)

        print(f"Generated {len(clue_guesses)} guesses for clue {clue.number} {clue.direction}:")
        guesses[clue.id] = clue_guesses

    clue_guesses = guesses[clue.id]

    if len(clue_guesses) == 0:
        print(f"No guesses left for clue: {clue.number} {clue.direction} - {clue.text}, backtracking...")
        guesses.pop(clue.id)

        if len(crossword_puzzle.completed_clues) > 0:
            clue_difficulties[clue.id] += 100
            crossword_puzzle.remove_answer(crossword_puzzle.completed_clues[-1])

        continue

    print(f"Guesses for clue {clue.number} {clue.direction}:")
    for guess in clue_guesses:
        print(f" - {guess.answer} (confidence: {guess.confidence_score}): {guess.explanation}")

    best_guess = max(clue_guesses, key=lambda g: g.confidence_score)

    try:
        crossword_puzzle.set_answer(clue, best_guess.answer)
        clue_guesses.remove(best_guess)

        print(f"Set answer for clue {clue.number} {clue.direction} to '{best_guess.answer}'")
    except Exception as e:
        clue_guesses.remove(best_guess)
        print(f"Error setting answer for clue {clue.number} {clue.direction}: {e}")

◼ _ _ _ _ 
_ _ _ _ _ 
_ _ _ _ _ 
_ _ _ _ _ 
_ _ _ _ _ 

=== Solver iteration ===
Known letters per clue:
  (1, 'across'): known=0, difficulty=37.5
  (1, 'down'): known=0, difficulty=43.75
  (2, 'down'): known=0, difficulty=37.5
  (3, 'down'): known=0, difficulty=43.75
  (4, 'down'): known=0, difficulty=46.25
  (5, 'across'): known=0, difficulty=50.0
  (5, 'down'): known=0, difficulty=41.25
  (6, 'across'): known=0, difficulty=43.75
  (7, 'across'): known=0, difficulty=41.25
  (8, 'across'): known=0, difficulty=46.25

Ranked incomplete clues (best first):
  1. 1 across (known=0, difficulty=37.5) - Turn into a pulp, as potatoes
  2. 2 down   (known=0, difficulty=37.5) - Throat-burningly pungent
  3. 7 across (known=0, difficulty=41.25) - Musician's twiddle
  4. 5 down   (known=0, difficulty=41.25) - Video surveillance system, briefly
  5. 1 down   (known=0, difficulty=43.75) - Like the haka
  6. 6 across (known=0, difficulty=43.75) - Reef organism
  7. 3 down   (known=0, difficulty=43.75